# 04 — NAFIM: Noise-Aware Fusion for Image–text Memes
**Goal:** one model that stays accurate whatever text source it gets at test time — gold captions, Surya OCR or EasyOCR — and on badly OCR'd memes.

**NAFIM components**
1. **CER-supervised reliability estimator** — predicts how trustworthy the meme text is (target = 1 − CER vs. gold) from the text embedding + 11 text-quality features.
2. **Reliability-gated fusion** — text features are scaled by the predicted reliability before fusion with the CLIP image feature.
3. **Text-source mixing** — each training sample randomly uses gold, Surya, EasyOCR or synthetically corrupted text.
4. **Consistency loss** — prediction on the noisy text is pulled towards the prediction on the gold text of the same meme.

**Trained here:** NAFIM and a **Mixed-text fusion** baseline (same fusion as before, trained on gold + Surya + EasyOCR, no gate/reliability/consistency).
**Reused from notebook 03 (no retraining):** Fusion trained on gold, Fusion trained on OCR.

**Requires:** notebooks 01–03 finished. Kernel **Python (internship1)**. **Runtime:** ~2.5–3 h, resumable.

## 1. Imports & config

In [ ]:
import os, re, gc, json, time, random, platform, unicodedata
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before importing torch
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
import transformers, sklearn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score
from scipy.stats import ttest_rel, spearmanr
from rapidfuzz.distance import Levenshtein
import matplotlib.pyplot as plt

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P, FEAT = f"{ROOT}/processed", f"{ROOT}/features"
NB02, NB03 = f"{ROOT}/results/02_finetuned", f"{ROOT}/results/03_ocr_eval"
RES = f"{ROOT}/results/04_nafim"
PRED = f"{RES}/preds"
os.makedirs(PRED, exist_ok=True)

DATASETS = ["bhm", "mimosa"]
TASKS = [("bhm", "label"), ("mimosa", "label"), ("mimosa", "label_bin")]
SEEDS = [42, 43, 44, 45, 46]
TEXT_MODELS = {"BanglaBERT": "csebuetnlp/banglabert", "XLM-R": "xlm-roberta-base"}

# text sources: index in the encoded tensor
REAL_SOURCES = {"gold": "text", "surya": "text_surya", "easyocr": "text_easyocr", "ocr": "text_ocr"}
SYNTH_K = 3                                   # synthetic corrupted versions of each gold caption
EVAL_SOURCES = ["gold", "surya", "easyocr"]  # robustness metrics = mean / worst over these
HIGH_CER = 0.5                               # "badly OCR'd" threshold (on Surya+fallback text)

# NAFIM loss weights and text-source sampling
LAMBDA_REL, LAMBDA_CONS = 0.5, 1.0
NAFIM_MIX = {"gold": 0.25, "surya": 0.25, "easyocr": 0.25, "synth": 0.25}
MIXED_MIX = {"gold": 1 / 3, "surya": 1 / 3, "easyocr": 1 / 3}
SYNTH_SEED = 0
EVAL_BS = 64

dev = "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| AMP:", AMP_DTYPE)
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 2**30:.2f} / {total / 2**30:.2f} GiB  (close other GPU programs if below ~6 GiB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def slug(s):
    return s.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")

## 2. Settings inherited from notebooks 02 & 03 (same encoder, hyperparameters, frozen word embeddings)

In [ ]:
cfg02 = json.load(open(f"{NB02}/config.json"))["text"]
cfg03 = json.load(open(f"{NB03}/config.json"))
MAX_LEN, BS = cfg02["max_len"], cfg02["batch_size"]
MAX_EPOCHS, PATIENCE = cfg02["max_epochs"], cfg02["patience"]
LR_ENC, LR_HEAD, WD = cfg02["lr_encoder"], cfg02["lr_head"], cfg02["weight_decay"]
WARMUP, DROPOUT = cfg02["warmup"], cfg02["dropout"]

BASE_MODEL = cfg03["selected_model"]                       # e.g. "Fusion XLM-R+CLIP"
ENC_KEY = BASE_MODEL.split(" ", 1)[1].replace("+CLIP", "")
FREEZE_WORD_EMB = "frozen" in str(cfg03.get("word_embeddings", "frozen"))
assert BASE_MODEL.startswith("Fusion"), "notebook 03 must have selected a fusion model"
print("Base fusion model:", BASE_MODEL, "| encoder:", ENC_KEY, "| word embeddings frozen:", FREEZE_WORD_EMB)
print({k: cfg02[k] for k in ["max_len", "batch_size", "max_epochs", "patience", "lr_encoder", "lr_head"]})

## 3. Load data, CLIP features, and build synthetic noisy captions

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[^\u0980-\u09FFa-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def cer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    return np.nan if not r else Levenshtein.distance(r, h) / len(r)

BN_LETTERS = [chr(c) for c in range(0x0985, 0x09BA)]
BN_SIGNS = list("\u09BE\u09BF\u09C0\u09C1\u09C2\u09C3\u09C7\u09C8\u09CB\u09CC\u09CD\u0982\u0983\u0981")
LATIN = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

def corrupt(text, rng):
    """OCR-like corruption: char deletion / substitution / insertion, dropped vowel signs,
    junk tokens (watermark-like) and dropped words. Rate ~ U(0.05, 0.45)."""
    rate = rng.uniform(0.05, 0.45)
    out = []
    for ch in str(text):
        u = rng.random()
        if u < rate * 0.30:
            continue                                            # deletion
        elif u < rate * 0.60:
            out.append(rng.choice(BN_LETTERS + LATIN))          # substitution
        elif u < rate * 0.75:
            out.append(ch); out.append(rng.choice(BN_LETTERS + LATIN))   # insertion
        elif u < rate and ch in BN_SIGNS:
            continue                                            # lost vowel sign / hasanta
        else:
            out.append(ch)
    words = "".join(out).split()
    if words and rng.random() < 0.2:
        words.pop(rng.randrange(len(words)))                    # dropped word
    if rng.random() < 0.3:
        junk = "".join(rng.choice(LATIN) for _ in range(rng.randint(3, 8)))
        words.insert(rng.randint(0, len(words)), junk)          # watermark-like junk
    return " ".join(words)

data = {}
for ds in DATASETS:
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    for c in REAL_SOURCES.values():
        assert c in df.columns, f"{ds}: missing column {c} - run ocr_gold_datasets.py"
        df[c] = df[c].astype(str)
    rng = random.Random(f"{SYNTH_SEED}-{ds}")
    for k in range(SYNTH_K):
        df[f"text_synth{k}"] = [corrupt(t, rng) for t in df["text"]]
    img = np.load(f"{FEAT}/{ds}_img.npy").astype(np.float32)
    assert len(df) == len(img)
    data[ds] = {"df": df, "img": torch.from_numpy(img)}

SOURCES = list(REAL_SOURCES) + [f"synth{k}" for k in range(SYNTH_K)]
SRC_COL = {**REAL_SOURCES, **{f"synth{k}": f"text_synth{k}" for k in range(SYNTH_K)}}
SRC_IDX = {s: i for i, s in enumerate(SOURCES)}

for ds in DATASETS:
    df = data[ds]["df"]
    for s in SOURCES:
        df[f"cer_{s}"] = 0.0 if s == "gold" else [cer(r, h) for r, h in zip(df["text"], df[SRC_COL[s]])]
    df.loc[df["text"].map(norm) == "", [f"cer_{s}" for s in SOURCES]] = np.nan
    print(ds, "| mean CER  " + "  ".join(f"{s}={df[f'cer_{s}'].mean():.3f}" for s in SOURCES))
print("\nExample synthetic corruption:\n gold :", data["bhm"]["df"]["text"].iloc[0], "\n synth:", data["bhm"]["df"]["text_synth0"].iloc[0])

## 4. Tokenize every text source + text-quality features + reliability targets

In [ ]:
tok = AutoTokenizer.from_pretrained(TEXT_MODELS[ENC_KEY])

def quality_features(texts):
    """11 cheap features computable for ANY text at test time (no gold needed)."""
    pieces = [len(x) for x in tok(list(texts), add_special_tokens=False)["input_ids"]]
    rows = []
    for t, npc in zip(texts, pieces):
        t = str(t); n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        bn = sum("\u0980" <= c <= "\u09FF" for c in t); lat = sum(c.isascii() and c.isalpha() for c in t)
        dig = sum(c.isdigit() for c in t); sp = sum(c.isspace() for c in t)
        pun = sum(c in ".,!?;:'\"-।()" for c in t)
        rows.append([np.log1p(len(t)), np.log1p(len(words)), float(len(t.strip()) == 0),
                     bn / n, lat / n, dig / n, max(n - bn - lat - dig - sp - pun, 0) / n, pun / n,
                     sum(len(w) == 1 for w in words) / nw, np.mean([len(w) for w in words]) if words else 0.0,
                     npc / nw])
    return np.array(rows, dtype=np.float32)
Q_NAMES = ["log_chars", "log_words", "empty", "bengali_ratio", "latin_ratio", "digit_ratio", "junk_ratio",
           "punct_ratio", "single_char_word_ratio", "mean_word_len", "subwords_per_word"]

def idx_of(df, s):
    return np.where(df["split"].values == s)[0]

for ds in DATASETS:
    df = data[ds]["df"]
    enc = [tok(df[SRC_COL[s]].tolist(), padding="max_length", truncation=True, max_length=MAX_LEN,
               return_tensors="pt") for s in SOURCES]
    data[ds]["ids"] = torch.stack([e["input_ids"] for e in enc])           # [S, N, L]
    data[ds]["mask"] = torch.stack([e["attention_mask"] for e in enc])
    Q = np.stack([quality_features(df[SRC_COL[s]].tolist()) for s in SOURCES])  # [S, N, 11]
    tr = idx_of(df, "train")
    train_src = [SRC_IDX[s] for s in ["gold", "surya", "easyocr"]] + [SRC_IDX[f"synth{k}"] for k in range(SYNTH_K)]
    mu = Q[train_src][:, tr].reshape(-1, Q.shape[-1]).mean(0)
    sd = Q[train_src][:, tr].reshape(-1, Q.shape[-1]).std(0) + 1e-6
    data[ds]["q"] = torch.from_numpy((Q - mu) / sd)
    rel = np.stack([np.clip(1 - df[f"cer_{s}"].values, 0, 1) for s in SOURCES])  # NaN where gold empty
    data[ds]["rel"] = torch.from_numpy(rel.astype(np.float32))
    print(ds, "ids", tuple(data[ds]["ids"].shape), "| q", tuple(data[ds]["q"].shape))

for ds, lab in TASKS:
    df = data[ds]["df"]
    np.save(f"{PRED}/{ds}__{lab}__ytrue.npy", df[lab].values[idx_of(df, "test")].astype(int))

## 5. Models: Mixed-text fusion baseline and NAFIM

In [ ]:
def build_encoder():
    enc = AutoModel.from_pretrained(TEXT_MODELS[ENC_KEY])
    if FREEZE_WORD_EMB:
        enc.get_input_embeddings().weight.requires_grad_(False)
    return enc

def mean_pool(hs, mask):
    m = mask.unsqueeze(-1).to(hs.dtype)
    return (hs * m).sum(1) / m.sum(1).clamp(min=1)

class FusionBaseline(nn.Module):
    """Identical to the notebook 02/03 fusion model (text mean-pool ⊕ projected CLIP → MLP)."""
    def __init__(self, n, img_dim):
        super().__init__()
        self.enc = build_encoder(); h = self.enc.config.hidden_size
        self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * h, h), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
    def forward(self, ids, mask, img, q=None):
        t = mean_pool(self.enc(input_ids=ids, attention_mask=mask).last_hidden_state, mask)
        return self.head(torch.cat([t, self.img_proj(img)], -1)), None

class NAFIM(nn.Module):
    """Reliability-gated fusion. r = sigmoid(MLP([text, quality features])) is trained to predict 1-CER;
    the text vector is multiplied by r before fusion, and r itself is given to the classifier."""
    def __init__(self, n, img_dim, q_dim, use_gate=True):
        super().__init__()
        self.enc = build_encoder(); h = self.enc.config.hidden_size
        self.use_gate = use_gate
        self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.rel_head = nn.Sequential(nn.Linear(h + q_dim, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * h + 1, h), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
    def forward(self, ids, mask, img, q):
        t = mean_pool(self.enc(input_ids=ids, attention_mask=mask).last_hidden_state, mask)
        r = torch.sigmoid(self.rel_head(torch.cat([t, q.to(t.dtype)], -1)))
        tg = r * t if self.use_gate else t
        return self.head(torch.cat([tg, self.img_proj(img), r], -1)), r.squeeze(-1)

## 6. Training (source mixing, reliability + consistency losses, model selection on mean val F1 over gold/Surya/EasyOCR)

In [ ]:
def class_weights(y, n):
    counts = np.bincount(y, minlength=n)
    return torch.tensor(len(y) / (n * np.maximum(counts, 1)), dtype=torch.float32, device=dev)

def sample_sources(n_items, mix, g):
    groups = list(mix)
    pick = torch.multinomial(torch.tensor([mix[k] for k in groups]), n_items, replacement=True, generator=g)
    synth = torch.randint(0, SYNTH_K, (n_items,), generator=g)
    out = [SRC_IDX[f"synth{int(synth[i])}"] if groups[p] == "synth" else SRC_IDX[groups[p]] for i, p in enumerate(pick.tolist())]
    return torch.tensor(out)

@torch.no_grad()
def predict(model, ds, src, idx):
    model.eval(); d = data[ds]; s = SRC_IDX[src]; probs, rels = [], []
    for i in range(0, len(idx), EVAL_BS):
        b = torch.from_numpy(idx[i:i + EVAL_BS])
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            lo, r = model(d["ids"][s, b].to(dev), d["mask"][s, b].to(dev), d["img"][b].to(dev), d["q"][s, b].to(dev))
        probs.append(torch.softmax(lo.float(), 1).cpu())
        rels.append(r.float().cpu() if r is not None else torch.full((len(b),), float("nan")))
    return torch.cat(probs).numpy(), torch.cat(rels).numpy()

def run_train(kind, ds, lab, seed, ckpt=False):
    set_seed(seed)
    d = data[ds]; df = d["df"]; y = df[lab].values.astype(np.int64); n = int(y.max()) + 1
    Y = torch.from_numpy(y)
    tr, va, te = [idx_of(df, s) for s in ("train", "val", "test")]
    model = (NAFIM(n, d["img"].shape[1], d["q"].shape[-1]) if kind == "NAFIM" else FusionBaseline(n, d["img"].shape[1])).to(dev)
    if ckpt:
        model.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    enc_ids = {id(p) for p in model.enc.parameters()}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    enc_params = [p for p in model.enc.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENC}, {"params": head_params, "lr": LR_HEAD}],
                            weight_decay=WD, foreach=False)
    steps_per_epoch = (len(tr) + BS - 1) // BS
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * steps_per_epoch * MAX_EPOCHS), steps_per_epoch * MAX_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    cw = class_weights(y[tr], n)
    g = torch.Generator().manual_seed(seed)
    mix = NAFIM_MIX if kind == "NAFIM" else MIXED_MIX
    trainable = [p for p in model.parameters() if p.requires_grad]

    best_f, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.from_numpy(tr)[torch.randperm(len(tr), generator=g)]
        for i in tqdm(range(0, len(perm), BS), desc=f"{kind} {ds}/{lab} s{seed} ep{ep + 1}", leave=False):
            b = perm[i:i + BS]; src = sample_sources(len(b), mix, g)
            yb = Y[b].to(dev)
            if kind == "NAFIM":     # one forward pass over [gold view ; noisy view]
                gold = torch.full_like(src, SRC_IDX["gold"])
                s2, b2 = torch.cat([gold, src]), torch.cat([b, b])
                ids, mask = d["ids"][s2, b2].to(dev), d["mask"][s2, b2].to(dev)
                img, q, rt = d["img"][b2].to(dev), d["q"][s2, b2].to(dev), d["rel"][s2, b2].to(dev)
            else:
                ids, mask = d["ids"][src, b].to(dev), d["mask"][src, b].to(dev)
                img, q = d["img"][b].to(dev), d["q"][src, b].to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                lo, r = model(ids, mask, img, q)
            lo = lo.float()
            if kind == "NAFIM":
                lg, lv = lo[:len(b)], lo[len(b):]
                ce = 0.5 * (F.cross_entropy(lg, yb, weight=cw) + F.cross_entropy(lv, yb, weight=cw))
                ok = ~torch.isnan(rt)
                rel_loss = F.mse_loss(r.float()[ok], rt[ok]) if ok.any() else lo.sum() * 0
                cons = F.kl_div(F.log_softmax(lv, -1), F.softmax(lg, -1).detach(), reduction="batchmean")
                loss = ce + LAMBDA_REL * rel_loss + LAMBDA_CONS * cons
            else:
                loss = F.cross_entropy(lo, yb, weight=cw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        f = np.mean([f1_score(y[va], predict(model, ds, s, va)[0].argmax(1), average="macro") for s in EVAL_SOURCES])
        if f > best_f:
            best_f, best_ep, bad = f, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    out = {s: predict(model, ds, s, te) for s in REAL_SOURCES}
    del model, opt, best_state, sch, scaler
    gc.collect(); torch.cuda.empty_cache()
    return best_f, best_ep, out, y[te]

## 7. Train Mixed-text baseline and NAFIM (long, resumable)

In [ ]:
MODELS = ["Fusion (mixed text)", "NAFIM"]
KIND = {"Fusion (mixed text)": "mixed", "NAFIM": "NAFIM"}
RUNS_CSV = f"{RES}/runs.csv"
done = set(pd.read_csv(RUNS_CSV)["train_id"]) if os.path.exists(RUNS_CSV) else set()
print("trainings already done:", len(done), "of", len(SEEDS) * len(TASKS) * len(MODELS))

for seed in SEEDS:
    for name in MODELS:
        for ds, lab in TASKS:
            train_id = f"{ds}__{lab}__{slug(name)}__s{seed}"
            if train_id in done:
                continue
            t0 = time.time()
            for ckpt in (False, True):            # on out-of-memory, retry once with gradient checkpointing
                try:
                    vf, ep, out, y_te = run_train(KIND[name], ds, lab, seed, ckpt=ckpt)
                    break
                except torch.cuda.OutOfMemoryError:
                    gc.collect(); torch.cuda.empty_cache()
                    if ckpt:
                        raise
                    print(f"  out of GPU memory on {train_id} - retrying with gradient checkpointing")
            mins = round((time.time() - t0) / 60, 2)
            rows = []
            for s, (p, r) in out.items():
                np.save(f"{PRED}/{train_id}__test-{s}.npy", p)
                if KIND[name] == "NAFIM":
                    np.save(f"{PRED}/{train_id}__test-{s}__rel.npy", r)
                rows.append({"train_id": train_id, "dataset": ds, "task": lab, "model": name, "seed": seed,
                             "test_text": s, "best_epoch": ep, "val_mean_f1": vf,
                             "test_f1": f1_score(y_te, p.argmax(1), average="macro"),
                             "test_acc": accuracy_score(y_te, p.argmax(1)), "minutes": mins,
                             "word_emb": "frozen" if FREEZE_WORD_EMB else "trainable"})
            pd.DataFrame(rows).to_csv(RUNS_CSV, mode="a", header=not os.path.exists(RUNS_CSV), index=False)
            done.add(train_id)
            print(f"{ds:7s} {lab:10s} {name:20s} s{seed} | " + "  ".join(f"{r['test_text']}={r['test_f1']:.4f}" for r in rows) + f"  ({mins} min)")

## 8. Main comparison — NAFIM vs. all fusion baselines (mean ± std over seeds)

In [ ]:
PRED03 = f"{NB03}/preds"
BASE_SLUG = slug(BASE_MODEL)
COMPARE = {  # display name -> function(ds, lab, seed, test_src) -> prediction file
    "Fusion (train gold)": lambda ds, lab, s, t: f"{PRED03}/{ds}__{lab}__{BASE_SLUG}__train-gold__s{s}__test-{t}.npy",
    "Fusion (train OCR)":  lambda ds, lab, s, t: f"{PRED03}/{ds}__{lab}__{BASE_SLUG}__train-ocr__s{s}__test-{t}.npy",
    "Fusion (mixed text)": lambda ds, lab, s, t: f"{PRED}/{ds}__{lab}__{slug('Fusion (mixed text)')}__s{s}__test-{t}.npy",
    "NAFIM":               lambda ds, lab, s, t: f"{PRED}/{ds}__{lab}__NAFIM__s{s}__test-{t}.npy",
}
METRICS = ["gold", "surya", "easyocr", "ocr", "mean(gold,surya,easyocr)", "worst(gold,surya,easyocr)", f"high-CER OCR (≥{HIGH_CER})"]

rows = []
for ds, lab in TASKS:
    df = data[ds]["df"]; te = idx_of(df, "test")
    y = np.load(f"{PRED}/{ds}__{lab}__ytrue.npy")
    hi = (df["cer_ocr"].values[te] >= HIGH_CER)
    for name, path in COMPARE.items():
        for s in SEEDS:
            files = {t: path(ds, lab, s, t) for t in REAL_SOURCES}
            if not all(os.path.exists(f) for f in files.values()):
                continue
            pr = {t: np.load(f).argmax(1) for t, f in files.items()}
            f1 = {t: f1_score(y, pr[t], average="macro") for t in REAL_SOURCES}
            ev = [f1[t] for t in EVAL_SOURCES]
            rows.append({"dataset": ds, "task": lab, "model": name, "seed": s, **f1,
                         METRICS[4]: np.mean(ev), METRICS[5]: np.min(ev),
                         METRICS[6]: f1_score(y[hi], pr["ocr"][hi], average="macro"), "n_high_cer": int(hi.sum())})
cmp = pd.DataFrame(rows)

def fmt(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f}"
order = list(COMPARE)
main = (cmp.groupby(["dataset", "task", "model"])[METRICS].agg(fmt)
           .reindex(pd.MultiIndex.from_tuples([(d, t, m) for d, t in TASKS for m in order]), fill_value="—"))
print("seeds per model:", cmp.groupby("model")["seed"].nunique().to_dict(), "| high-CER test memes:",
      cmp.drop_duplicates(["dataset", "task"]).set_index(["dataset", "task"])["n_high_cer"].to_dict())
main

## 9. Is NAFIM significantly better? (paired t-test over seeds, NAFIM − baseline)

In [ ]:
sig = []
for (ds, lab), grp in cmp.groupby(["dataset", "task"]):
    nf = grp[grp.model == "NAFIM"].set_index("seed")
    for base in [m for m in order if m != "NAFIM"]:
        bs = grp[grp.model == base].set_index("seed")
        common = nf.index.intersection(bs.index)
        if len(common) < 2:
            continue
        for m in METRICS:
            d = nf.loc[common, m] - bs.loc[common, m]
            sig.append({"dataset": ds, "task": lab, "vs": base, "metric": m, "delta": d.mean(),
                        "p_value": ttest_rel(nf.loc[common, m], bs.loc[common, m]).pvalue, "n_seeds": len(common)})
sig_df = pd.DataFrame(sig, columns=["dataset", "task", "vs", "metric", "delta", "p_value", "n_seeds"])
sig_df["significant(p<0.05)"] = sig_df["p_value"] < 0.05
sig_df["result"] = np.where(sig_df["significant(p<0.05)"], np.where(sig_df["delta"] > 0, "NAFIM better", "NAFIM worse"), "n.s.")
sig_df.pivot_table(index=["dataset", "task", "vs"], columns="metric", values="delta", aggfunc="first")[METRICS].round(4)

In [ ]:
sig_df.pivot_table(index=["dataset", "task", "vs"], columns="metric", values="result", aggfunc="first")[METRICS]

## 10. Does the reliability estimator work? (predicted reliability vs. true 1 − CER, test set)

In [ ]:
rel_rows, scatter = [], {}
for ds, lab in TASKS:
    df = data[ds]["df"]; te = idx_of(df, "test")
    for s in SEEDS:
        for src in REAL_SOURCES:
            f = f"{PRED}/{ds}__{lab}__NAFIM__s{s}__test-{src}__rel.npy"
            if not os.path.exists(f):
                continue
            r = np.load(f); true = 1 - df[f"cer_{src}"].values[te]; ok = ~np.isnan(true)
            if src != "gold" and ok.sum() > 10:
                rho = spearmanr(r[ok], np.clip(true[ok], 0, 1)).correlation
                rel_rows.append({"dataset": ds, "task": lab, "seed": s, "test_text": src, "spearman_rho": rho,
                                 "mean_reliability": r.mean()})
            if s == SEEDS[0] and src == "ocr":
                scatter[(ds, lab)] = (np.clip(true[ok], 0, 1), r[ok])
rel_df = pd.DataFrame(rel_rows)
if len(rel_df):
    display(rel_df.groupby(["dataset", "task", "test_text"])[["spearman_rho", "mean_reliability"]].mean().round(3))

fig, axes = plt.subplots(1, len(scatter), figsize=(4.5 * max(len(scatter), 1), 4), squeeze=False)
for ax, ((ds, lab), (t, r)) in zip(axes[0], scatter.items()):
    ax.scatter(t, r, s=6, alpha=0.4)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_title(f"{ds}/{lab} (OCR text, seed {SEEDS[0]})"); ax.set_xlabel("true reliability (1 − CER)")
axes[0][0].set_ylabel("NAFIM predicted reliability")
fig.tight_layout(); fig.savefig(f"{RES}/reliability_scatter.png", dpi=200); plt.show()

## 11. Figure — mean and worst-case F1 across text sources

In [ ]:
fig, axes = plt.subplots(1, len(TASKS), figsize=(5 * len(TASKS), 4), sharey=True)
axes = np.atleast_1d(axes)
x = np.arange(len(order)); w = 0.38
for ax, (ds, lab) in zip(axes, TASKS):
    sub = cmp[(cmp.dataset == ds) & (cmp.task == lab)]
    for j, (m, lbl) in enumerate([(METRICS[4], "mean over text sources"), (METRICS[5], "worst text source")]):
        mu = [sub[sub.model == o][m].mean() for o in order]; sd = [sub[sub.model == o][m].std(ddof=1) for o in order]
        ax.bar(x + (j - 0.5) * w, mu, w, yerr=sd, capsize=3, label=lbl)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
    ax.set_title(f"{ds} / {lab}"); ax.grid(axis="y", alpha=0.3)
    lo = np.nanmin(sub[[METRICS[5]]].values) if len(sub) else 0.5
    ax.set_ylim(max(0, lo - 0.05), None)
axes[0].set_ylabel("macro-F1"); axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{RES}/robustness_bars.png", dpi=200); plt.show()

## 12. Save results + config

In [ ]:
cmp.to_csv(f"{RES}/comparison_per_seed.csv", index=False)
main.to_csv(f"{RES}/main_table.csv")
sig_df.to_csv(f"{RES}/nafim_vs_baselines_ttest.csv", index=False)
if len(rel_df):
    rel_df.to_csv(f"{RES}/reliability_spearman.csv", index=False)
config = {
    "base_model": BASE_MODEL, "encoder": TEXT_MODELS[ENC_KEY], "word_embeddings_frozen": FREEZE_WORD_EMB,
    "tasks": TASKS, "seeds": SEEDS, "hyperparameters_from_nb02": cfg02,
    "nafim": {"loss": "0.5*(CE_gold + CE_view) + LAMBDA_REL*MSE(r, 1-CER) + LAMBDA_CONS*KL(p_gold || p_view)",
              "lambda_rel": LAMBDA_REL, "lambda_cons": LAMBDA_CONS, "text_source_mix": NAFIM_MIX,
              "gate": "text_vec * r, r appended to fused vector", "quality_features": Q_NAMES},
    "mixed_baseline": {"text_source_mix": MIXED_MIX, "loss": "CE"},
    "synthetic_noise": {"versions_per_caption": SYNTH_K, "seed": SYNTH_SEED,
                        "ops": "char deletion/substitution/insertion, dropped vowel signs, dropped word, junk token; rate~U(0.05,0.45)"},
    "model_selection": "best epoch by mean validation macro-F1 over gold/Surya/EasyOCR text",
    "baselines_from_nb03": ["Fusion (train gold)", "Fusion (train OCR)"],
    "note": "nb03 baselines were selected on validation F1 of their own training text source",
    "high_cer_threshold": HIGH_CER,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__,
                 "gpu": torch.cuda.get_device_name(0)},
}
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("saved to", RES)